TODO:
~~Use numpy instead of 2d list~~
~~Clean up code~~
~~Finish enviorment and move code to seperate python file instead of juypter notebook (conflict errors with jupyter can be horrid)~~
~~Create Q-Learning agent (NO DeepL)~~
Create DeepL Agent (DQN)
Train with limited action to suggest better placement (more 4 line clears)
Add experience replay
Adjust training values until satisfactory

FOR PRESENTATION:
Train to get data
Create graphs for data (reward, score, average steps, etc)
Add imageIO rendering so video can be created of tutorial runs
https://www.tensorflow.org/agents/tutorials/10_checkpointer_policysaver_tutorial

FUTURE TODO(?)
Increase board size
Add different agent
Try different networks (actor critic, etc)
Compete against human
Work with time instead of step

In [ ]:
pip install --upgrade tensorflow-probability

In [1]:
# Imports
# pyright: ignore[reportMissingImports]

import os
import sys
# Keep using keras-2 (tf-keras) rather than keras-3 (keras).
os.environ['TF_USE_LEGACY_KERAS'] = '1'
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import abc
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np

from tf_agents.specs import array_spec
from tf_agents.specs import tensor_spec
from tf_agents.networks import network

from tf_agents.policies import py_policy
from tf_agents.policies import random_py_policy
from tf_agents.policies import scripted_py_policy

from tf_agents.policies import tf_policy
from tf_agents.policies import random_tf_policy
from tf_agents.policies import actor_policy
from tf_agents.policies import q_policy
from tf_agents.policies import greedy_policy

from tf_agents.trajectories import time_step as ts

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

from tf_agents.environments import py_environment
from tf_agents.environments import tf_environment
from tf_agents.environments import tf_py_environment
from tf_agents.environments import utils
from tf_agents.specs import array_spec
from tf_agents.environments import wrappers
from tf_agents.environments import suite_gym
from tf_agents.trajectories import time_step as ts

2026-03-03 11:50:08.259619: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-03 11:50:08.310146: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 11:50:10.496384: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 11:50:13.926034: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [ ]:
"""
Random Action Agent
Author: Nathan Delcampo
Date: 1/13/2026
Last Modifed: 1/27/2026
Python Version: 3.11.14

DESC: Random Action Agent that works off of tetris implementaiton
"""

import sys, importlib
from tetris import Tetris # self tetris implementation
importlib.reload(sys.modules['tetris'])
import pygame

# Action_spec for 5 possible actions
# RIGHT LEFT COUNTERCLOCKWISE CLOCKWISE NOOP
action_spec = array_spec.BoundedArraySpec(
    shape=(),
    dtype=np.int32,
    minimum=0,
    maximum=4,
    name='action'
)

# Setup tensorflow for random actions
input_tensor_spec = tensor_spec.TensorSpec((2,), tf.float32)
time_step_spec = ts.time_step_spec(input_tensor_spec)

my_random_tf_policy = random_tf_policy.RandomTFPolicy(
    action_spec=action_spec, time_step_spec=time_step_spec)
observation = tf.ones(time_step_spec.observation.shape)
time_step = ts.restart(observation)

# Setup Pygame
RED = (255, 0, 0)
BLACK = (0, 0, 0)

tetris = Tetris(_render=False)
pygame.init()
screen = pygame.display.set_mode((300, 700))
pygame.event.clear()
running = True

# Main Loop
while running:

    # Wait for pygame event
    event = pygame.event.wait()

    # Only respond to key presses
    if event.type == pygame.quit:
        running = False
        pygame.quit()
        sys.exit()
    elif event.type == pygame.KEYDOWN:
        if event.key == pygame.K_ESCAPE:
            running = False
            pygame.quit()
            sys.exit()
        else:

            # Get action from random agent and apply
            action = my_random_tf_policy.action(time_step)
            match action[0]:
                case 0:
                    print("RIGHT")
                    tetris.step("RIGHT")
                case 1:
                    print("LEFT")
                    tetris.step("LEFT")
                case 2:
                    print("CC")
                    tetris.step("COUNTERCLOCKWISE")
                case 3:
                    print("C")
                    tetris.step("CLOCKWISE")
                case 4:
                    print("N")
                    tetris.step("NOOP")
            
            # Get board copy for rendering
            board = tetris.get_board_copy()
            
            # Draw screen
            screen.fill(BLACK)

            for row_index, row in enumerate(board):
                for col_index, tile_value in enumerate(row):
                    # Calculate the position for the current tile
                    x = col_index * 50
                    y = row_index * 50

                    if (tile_value) == 0:
                        color = BLACK
                    else:
                        color = RED

                    # Draw the rectangle
                    pygame.draw.rect(screen, color, (x, y, 50, 50))

            print(tetris._get_observation_spec())

            # Update the display
            pygame.display.flip()

In [14]:
import sys, importlib
from tetris import Tetris # self tetris implementation
importlib.reload(sys.modules['tetris'])
import pygame

tetris = Tetris(_render=True)
userinput = ""

while (userinput != "STOP"):
    userinput = input()

    match userinput:
        case "r":
            print("RIGHT")
            tetris.step("RIGHT")
        case "l":
            print("LEFT")
            tetris.step("LEFT")
        case "cc":
            print("CC")
            tetris.step("COUNTERCLOCKWISE")
        case "c":
            print("C")
            tetris.step("CLOCKWISE")
        case "n":
            print("N")
            tetris.step("NOOP")
        case "h":
            print("HD")
            tetris.step("HARD_DROP")

N
Doing: NOOP
[0. 0. 0. 0. 0. 0.]
[0. 0. 2. 0. 0. 0.]
[0. 0. 2. 0. 0. 0.]
[0. 0. 2. 2. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]

N
Doing: NOOP
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 2. 0. 0. 0.]
[0. 0. 2. 0. 0. 0.]
[0. 0. 2. 2. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]



In [2]:
"""
Tetris TF-Agents PyEnvironment Wrapper
Wraps the Step Tetris implementation for use with TensorFlow Q-Learning (DQN).
"""

import numpy as np
import tensorflow as tf

from tf_agents.environments import py_environment
from tf_agents.specs import array_spec
from tf_agents.trajectories import time_step as ts

from tetris import Tetris


# Reward / feature helpers
def _count_holes(board: np.ndarray) -> int:
    """
    A hole is any empty cell (0) that has at least one filled cell (1) above
    it in the same column.
    """
    holes = 0
    rows, cols = board.shape
    for col in range(cols):
        block_found = False
        for row in range(rows):
            if board[row, col] == 1:
                block_found = True
            elif block_found:
                holes += 1
    return holes


def _aggregate_height(board: np.ndarray) -> int:
    """Sum of the height of each column (highest filled cell from bottom)."""
    rows, cols = board.shape
    total = 0
    for col in range(cols):
        for row in range(rows):
            if board[row, col] == 1:
                total += rows - row  # height from bottom
                break
    return total


def _bumpiness(board: np.ndarray) -> int:
    """Sum of absolute height differences between adjacent columns."""
    rows, cols = board.shape
    heights = []
    for col in range(cols):
        h = 0
        for row in range(rows):
            if board[row, col] == 1:
                h = rows - row
                break
        heights.append(h)
    return sum(abs(heights[i] - heights[i + 1]) for i in range(cols - 1))


# Environment

# Maps possible actions to integers
ACTION_MAP = {
    0: "LEFT",
    1: "RIGHT",
    2: "COUNTERCLOCKWISE",
    3: "CLOCKWISE",
    4: "NOOP",
    5: "HARD_DROP",
}

# How many extra features are appended after the flattened board
# [next_piece_id (1), lines_cleared_this_step (1)]
_EXTRA_FEATURES = 2


class TetrisEnv(py_environment.PyEnvironment):
    """
    TensorFlow PyEnvironment wrapper for the Tetris implementation.

    Observation
    -----------
    A flat float32 vector of length (rows * cols + _EXTRA_FEATURES):
        - Flattened board  (0.0 = empty, 1.0 = filled)
        - next_piece id    (normalised to [0, 1])
        - lines_cleared_this_step (raw int cast to float)

    Action
    ------
    Discrete integer in [0, 5] mapping to LEFT / RIGHT / CCW / CW / NOOP / HARD_DROP.

    Reward
    ------
    A shaped reward that encourages line clears and penalises building up holes,
    height, and bumpiness:

        reward = (lines_cleared² × 10)
               - 0.51 × aggregate_height
               - 0.36 × holes
               - 0.18 × bumpiness
               + 0.01  (small living bonus per step)

    Episode termination
    -------------------
    The episode ends when the Tetris game is over, i.e.
    a newly spawned piece immediately collides.
    """

    # Reward shaping weights (Dellacherie-style)
    w_height   = 0.51
    w_holes    = 0.36
    w_bumpiness    = 0.18
    w_lines    = 10.0
    w_time_alive = 0.01

    def __init__(self, rows: int = 14, cols: int = 6, render: bool = False):
        super().__init__()

        self._rows = rows
        self._cols = cols
        self._render = render

        obs_size = rows * cols + _EXTRA_FEATURES
        self._observation_spec = array_spec.BoundedArraySpec(
            shape=(obs_size,),
            dtype=np.float32,
            minimum=0.0,
            maximum=float(max(rows, cols, 6)),  # generous upper bound
            name="observation",
        )

        self._action_spec = array_spec.BoundedArraySpec(
            shape=(),
            dtype=np.int32,
            minimum=0,
            maximum=len(ACTION_MAP) - 1,
            name="action",
        )

        # Internal game instance
        self._game = Tetris(_rows=rows, _cols=cols, _render=render)
        self._episode_ended = False
        self._prev_score = 0

    # PyEnvironment interface

    def observation_spec(self):
        return self._observation_spec

    def action_spec(self):
        return self._action_spec

    def _reset(self):
        self._game.reset()
        self._episode_ended = False
        self._prev_score = 0
        return ts.restart(self._get_observation())

    def _step(self, action: int):
        if self._episode_ended:
            # Auto-reset at the start of a new episode
            return self._reset()

        score_before = self._game.score
        board_before = np.copy(self._game.board)

        # Get action from map and do step
        action_str = ACTION_MAP[int(action)]
        self._game.step(action_str)

        # Check for game over
        score_after = self._game.score
        game_over = (score_after < score_before) or self._game_over_detected(board_before)

        # Lines cleared this step (score delta, clamped)
        lines_cleared = max(0, score_after - score_before)

        # Shaped reward
        board_now = self._game.board
        reward = (
            self.w_lines * (lines_cleared ** 2)
            - self.w_height * _aggregate_height(board_now)
            - self.w_holes  * _count_holes(board_now)
            - self.w_bumpiness  * _bumpiness(board_now)
            + self.w_time_alive
        )

        obs = self._get_observation(lines_cleared=lines_cleared)

        if game_over:
            self._episode_ended = True
            return ts.termination(obs, reward=float(reward))
        else:
            return ts.transition(obs, reward=float(reward), discount=0.99)

    # Helpers functions
    def _game_over_detected(self, board_before: np.ndarray) -> bool:
        """
        Heuristic: if the board is now emptier than before, the game reset.
        (Tetris._game_over() calls self.reset() which zeros the board.)
        """
        filled_before = int(np.sum(board_before))
        filled_after  = int(np.sum(self._game.board))
        # A full reset wipes the board; fewer than half the blocks remaining
        # is a strong signal.
        return filled_after < filled_before // 2 and filled_before > 4

    def _get_observation(self, lines_cleared: int = 0) -> np.ndarray:
        """Build the flat observation vector."""
        flat_board = self._game.board.flatten().astype(np.float32)
        extras = np.array(
            [
                self._game.next_piece / 6.0,   # normalised piece id [0,1]
                float(lines_cleared),
            ],
            dtype=np.float32,
        )
        return np.concatenate([flat_board, extras])

    def render(self):
        self._game._render()


In [3]:
"""
Tetris DQN Training Loop
========================
Trains a DQN agent on TetrisEnv, periodically saves the policy and
writes training metrics to both a CSV log and TensorBoard summaries.

Directory layout after running:
    checkpoints/          <- tf_agents Checkpointer (full training state)
    saved_policy/         <- SavedModel export of the greedy policy
    logs/
        training_log.csv  <- per-iteration CSV (easy to load in pandas)
        tb/               <- TensorBoard event files

Usage (Jupyter):
    from train import CONFIG, train
    train(CONFIG)                          # use defaults
    train({**CONFIG, "iterations": 5000})  # override specific values

Usage (CLI):
    python train.py

TensorBoard:
    tensorboard --logdir logs/tb

Resuming:
    The Checkpointer automatically restores from checkpoints/ on the next run.
"""

import csv
import datetime
from pathlib import Path

import numpy as np
import tensorflow as tf

from tf_agents.agents.dqn import dqn_agent
from tf_agents.environments import tf_py_environment
from tf_agents.networks import sequential
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.policies import policy_saver, random_tf_policy
from tf_agents.utils import common
from tf_agents.trajectories import trajectory

# ---------------------------------------------------------------------------
# Config — edit values here or override at call-time (see Usage above)
# ---------------------------------------------------------------------------

CONFIG = {
    # Environment
    "rows": 14,
    "cols": 6,
    # Training schedule
    "iterations": 50_000,
    "initial_collect": 1_000,
    "collect_per_iter": 1,
    # Replay buffer
    "buffer_capacity": 50_000,
    "batch_size": 64,
    # Network
    "fc_layers": (128, 128),
    # Agent / optimiser
    "learning_rate": 1e-3,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "epsilon_decay": 50_000,
    "target_update_period": 500,
    # Logging / saving
    "log_interval": 200,
    "eval_interval": 1_000,
    "eval_episodes": 5,
    "save_interval": 5_000,
    "checkpoint_dir": "checkpoints",
    "policy_dir": "saved_policy",
    "log_dir": "logs",
}


# Helpers fucntions

class _Cfg:
    """Wraps a plain dict so values are accessible as attributes."""
    def __init__(self, d):
        self.__dict__.update(d)


def build_q_network(obs_spec, action_spec, fc_layers):
    num_actions = action_spec.maximum - action_spec.minimum + 1
    layers = []
    for units in fc_layers:
        layers.append(
            tf.keras.layers.Dense(
                units,
                activation="relu",
                kernel_initializer=tf.keras.initializers.VarianceScaling(
                    scale=2.0, mode="fan_in", distribution="truncated_normal"
                ),
            )
        )
    layers.append(
        tf.keras.layers.Dense(
            num_actions,
            activation=None,
            kernel_initializer=tf.keras.initializers.RandomUniform(
                minval=-0.03, maxval=0.03
            ),
        )
    )
    return sequential.Sequential(layers)


def collect_step(env, policy, buffer):
    time_step = env.current_time_step()
    action_step = policy.action(time_step)
    next_time_step = env.step(action_step.action)
    traj = trajectory.from_transition(time_step, action_step, next_time_step)
    buffer.add_batch(traj)


def run_eval(eval_env, policy, num_episodes):
    total_rewards = []
    for _ in range(num_episodes):
        time_step = eval_env.reset()
        ep_reward = 0.0
        while not time_step.is_last():
            action_step = policy.action(time_step)
            time_step = eval_env.step(action_step.action)
            ep_reward += float(time_step.reward[0])
        total_rewards.append(ep_reward)
    return float(np.mean(total_rewards))



# Main training function

def train(config: dict = None):
    """
    Run the DQN training loop.

    Args:
        config: dict of hyperparameters. Defaults to CONFIG defined above.
                Pass {**CONFIG, "key": new_value} to override specific values.
    """
    cfg = _Cfg(config if config is not None else CONFIG)

    print("\n=== Tetris DQN Training ===")
    print(f"Started: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}\n")

    # Environments
    train_py_env = TetrisEnv()
    eval_py_env  = TetrisEnv()
    train_env    = tf_py_environment.TFPyEnvironment(train_py_env)
    eval_env     = tf_py_environment.TFPyEnvironment(eval_py_env)

    obs_spec    = train_env.observation_spec()
    action_spec = train_env.action_spec()

    # Q-Network + Agent
    q_net = build_q_network(obs_spec, action_spec, cfg.fc_layers)
    train_step_counter = tf.Variable(0, trainable=False, dtype=tf.int64)

    # Growth Rate
    epsilon = tf.compat.v1.train.polynomial_decay(
        learning_rate=cfg.epsilon_start,
        global_step=train_step_counter,
        decay_steps=cfg.epsilon_decay,
        end_learning_rate=cfg.epsilon_end,
    )

    # Agent
    agent = dqn_agent.DqnAgent(
        train_env.time_step_spec(),
        action_spec,
        q_network=q_net,
        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate),
        td_errors_loss_fn=common.element_wise_squared_loss,
        epsilon_greedy=epsilon,
        target_update_period=cfg.target_update_period,
        gamma=cfg.gamma,
        train_step_counter=train_step_counter,
    )
    agent.initialize()

    # Replay buffer
    replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
        data_spec=agent.collect_data_spec,
        batch_size=train_env.batch_size,
        max_length=cfg.buffer_capacity,
    )
    dataset  = replay_buffer.as_dataset(
        num_parallel_calls=3, sample_batch_size=cfg.batch_size, num_steps=2
    ).prefetch(3)
    iterator = iter(dataset)

    # Random Policy for buffer
    random_policy  = random_tf_policy.RandomTFPolicy(
        train_env.time_step_spec(), action_spec
    )
    collect_policy = agent.collect_policy
    eval_policy    = agent.policy

    # Checkpointer
    checkpoint_dir = Path(cfg.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpointer = common.Checkpointer(
        ckpt_dir=str(checkpoint_dir),
        max_to_keep=3,
        agent=agent,
        policy=agent.policy,
        replay_buffer=replay_buffer,
        global_step=train_step_counter,
    )
    checkpointer.initialize_or_restore()
    print(f"Checkpoint dir : {checkpoint_dir.resolve()}")

    # Policy saver
    policy_dir = Path(cfg.policy_dir)
    policy_dir.mkdir(parents=True, exist_ok=True)
    print(f"Policy dir     : {policy_dir.resolve()}")

    # CSV + TensorBoard logging
    log_dir = Path(cfg.log_dir)
    (log_dir / "tb").mkdir(parents=True, exist_ok=True)
    summary_writer = tf.summary.create_file_writer(str(log_dir / "tb"))

    csv_path   = log_dir / "training_log.csv"
    csv_exists = csv_path.exists()
    csv_file   = open(csv_path, "a", newline="")
    csv_writer = csv.DictWriter(
        csv_file,
        fieldnames=["iteration", "loss", "epsilon", "eval_reward", "timestamp"],
    )
    if not csv_exists:
        csv_writer.writeheader()

    print(f"CSV log        : {csv_path.resolve()}")
    print(f"TensorBoard    : {(log_dir / 'tb').resolve()}\n")

    # Initial random collection
    start_step = int(train_step_counter.numpy())
    if start_step == 0:
        print(f"Collecting {cfg.initial_collect} random steps …")
        train_env.reset()
        for _ in range(cfg.initial_collect):
            collect_step(train_env, random_policy, replay_buffer)
        print("Initial collection done.\n")
    else:
        print(f"Resuming from step {start_step}.\n")

    # Training loop
    train_env.reset()
    loss_history = []

    for _ in range(cfg.iterations):
        for _ in range(cfg.collect_per_iter):
            collect_step(train_env, collect_policy, replay_buffer)

        experience, _ = next(iterator)
        train_loss = agent.train(experience).loss
        step = int(train_step_counter.numpy())
        loss_history.append(float(train_loss))

        if step % cfg.log_interval == 0:
            avg_loss = float(np.mean(loss_history[-cfg.log_interval:]))
            eps_val  = float(epsilon())
            print(f"[{step:>7}]  loss={avg_loss:.4f}  ε={eps_val:.3f}")
            with summary_writer.as_default():
                tf.summary.scalar("train/loss",    avg_loss, step=step)
                tf.summary.scalar("train/epsilon", eps_val,  step=step)

        if step % cfg.eval_interval == 0:
            eval_reward = run_eval(eval_env, eval_policy, cfg.eval_episodes)
            print(f"[{step:>7}]  *** eval reward = {eval_reward:.2f} ***")
            with summary_writer.as_default():
                tf.summary.scalar("eval/mean_reward", eval_reward, step=step)
            csv_writer.writerow({
                "iteration":   step,
                "loss":        f"{float(np.mean(loss_history[-cfg.eval_interval:])):.6f}",
                "epsilon":     f"{float(epsilon()):.4f}",
                "eval_reward": f"{eval_reward:.4f}",
                "timestamp":   datetime.datetime.now().isoformat(),
            })
            csv_file.flush()

        if step % cfg.save_interval == 0 and step > 0:
            checkpointer.save(global_step=train_step_counter)
            tf.saved_model.save(eval_policy, str(policy_dir))
            print(f"[{step:>7}]  Saved checkpoint + policy.")

    # Final save
    checkpointer.save(global_step=train_step_counter)
    tf.saved_model.save(eval_policy, str(policy_dir))
    csv_file.close()

    final_reward = run_eval(eval_env, eval_policy, cfg.eval_episodes * 2)
    print(f"\nTraining complete — final eval reward: {final_reward:.2f}")
    print(f"Policy saved to : {policy_dir.resolve()}")


In [ ]:
train(CONFIG)

In [7]:
from tf_agents.agents.dqn import dqn_agent
from tf_agents.environments import tf_py_environment
from tf_agents.utils import common
import tensorflow as tf

# Rebuild the env + agent 
env = tf_py_environment.TFPyEnvironment(TetrisEnv(rows=14, cols=6, render=True))
q_net = build_q_network(env.observation_spec(), env.action_spec(), (128, 128))
train_step_counter = tf.Variable(0, trainable=False, dtype=tf.int64)

agent = dqn_agent.DqnAgent(
    env.time_step_spec(),
    env.action_spec(),
    q_network=q_net,
    optimizer=tf.keras.optimizers.Adam(1e-3),
    td_errors_loss_fn=common.element_wise_squared_loss,
    train_step_counter=train_step_counter,
)
agent.initialize()

# Restore weights from checkpoint
checkpointer = common.Checkpointer(
    ckpt_dir="checkpoints",
    agent=agent,
    global_step=train_step_counter,
)
checkpointer.initialize_or_restore()
print(f"Restored from step {int(train_step_counter.numpy())}")

# Run eval
mean_reward = run_eval(env, agent.policy, num_episodes=1)
print(f"Mean reward: {mean_reward:.2f}")

Restored from step 260000
Doing: LEFT
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 2. 0. 0. 0. 0.]
[0. 2. 0. 0. 0. 0.]
[2. 2. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]

Doing: LEFT
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 2. 0. 0. 0. 0.]
[0. 2. 0. 0. 0. 0.]
[2. 2. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]

Doing: LEFT
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 2. 0. 0. 0. 0.]
[0. 2. 0. 0. 0. 0.]
[2. 2. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]

Doing: LEFT
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]
[0.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("logs/training_log.csv")

# Loss
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["iteration"], df["loss"])
ax.set_title("Training Loss")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
plt.tight_layout()
plt.savefig("loss.png")
plt.show()

# Eval reward
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["iteration"], df["eval_reward"], color="green")
ax.set_title("Eval Reward")
ax.set_xlabel("Step")
ax.set_ylabel("Mean Reward")
plt.tight_layout()
plt.savefig("reward.png")
plt.show()

# Epsilon decay
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["iteration"], df["epsilon"], color="orange")
ax.set_title("Epsilon (Exploration)")
ax.set_xlabel("Step")
ax.set_ylabel("ε")
plt.tight_layout()
plt.savefig("exploration.png")
plt.show()